# New Direct Inversion in Iterative Space (DIIS)

This method has been extensively used to solve self-consistent field (SCF) problems in the fields of quantum chemistry and physics. In this tutorial, we employ this method to accelerate the solving of fixed-point problems.

It should be noted that one potential issue with this method is that non-negative parameters cannot be guaranteed during optimization in the conventional DIIS approach. Although this issue can be addressed by explicitly introducing constraints to the linear combination coefficients, key concepts in DIIS, numerical issues may still arise, such as singular matrices or convergence problems.

In [1]:
from setup import prepare_argument_dict, prepare_grid_and_dens, print_results

from horton_part import (
    LinearISAWPart,
    lstsq_spsolver,
)

In DIIS method, the combination coefficients are determined by solving a least-square problem.
The default method is `lstsq_spsolver`, which computes the inversion of the matrix directly.
One can also use other least-square solver and availiable methods are `lstsq_solver_dyn`, `lstsq_solver_with_extra_constr`.

In [2]:
def run_diis(diis_method):
    """Self-consistent solver."""
    mol, grid, rho = prepare_grid_and_dens("data/h2o.fchk")
    kwargs = prepare_argument_dict(mol, grid, rho)
    kwargs["solver"] = "diis"
    print("*" * 80)
    print(f"Results with {diis_method.__name__}".center(80, " "))
    print("*" * 80)
    kwargs["solver_options"] = {"diis_size": 8, "lstsq_solver": diis_method}
    part = LinearISAWPart(**kwargs)
    part.do_all()
    print_results(part)
    print()

In [3]:
run_diis(lstsq_spsolver)

The number of electrons: 10.000003764139395
Coordinates of the atoms 
 [[ 0.     0.     0.224]
 [-0.     1.457 -0.896]
 [-0.    -1.457 -0.896]]
Atomic numbers of the atom 
 [8 1 1]
********************************************************************************
                          Results with lstsq_spsolver                           
********************************************************************************

Information of integral grids.
--------------------------------------------------------------------------------
Grid size of molecular grid: 18460
************************************ Atom 0 ************************************
   |-- Radial grid size: 120
   |-- Angular grid sizes: 
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18
          18 18 18 18 18 18 26 26 26 26 38 38 38 38 38 50 50 50 50 50
          86 86 110 110 110 110 170 194 194 434 590 5

/Users/runner/work/horton-part/horton-part/src/horton_part/algo/diis.py:163: MatrixRankWarning: Matrix is exactly singular
  sol = spsolve(B, rhs)


       19   2.37333e-04   3.96492e-02
       20   2.01386e-04   3.96168e-02
       21   1.71470e-04   3.95926e-02
       22   1.46497e-04   3.95743e-02
       23   1.25616e-04   3.95606e-02
       24   1.07757e-04   3.95501e-02
       25   9.28412e-05   3.95422e-02
       26   8.01292e-05   3.95361e-02
       27   6.93228e-05   3.95315e-02
       28   6.00894e-05   3.95279e-02
       29   5.22003e-05   3.95251e-02
       30   4.53971e-05   3.95230e-02
       31   3.95676e-05   3.95214e-02
       32   3.45304e-05   3.95201e-02
       33   3.01929e-05   3.95191e-02
       34   2.64301e-05   3.95183e-02
       35   2.31553e-05   3.95177e-02
       36   2.03205e-05   3.95173e-02
       37   1.78527e-05   3.95169e-02
       38   1.56925e-05   3.95166e-02
       39   1.38125e-05   3.95164e-02
       40   1.21672e-05   3.95162e-02
       41   1.07261e-05   3.95161e-02
       42   9.46283e-06   3.95160e-02
       43   8.35468e-06   3.95159e-02
       44   7.37965e-06   3.95158e-02
       45   

In [4]:
def run_cdiis(**solver_options):
    """Self-consistent solver."""
    mol, grid, rho = prepare_grid_and_dens("data/h2o.fchk")
    kwargs = prepare_argument_dict(mol, grid, rho)
    kwargs["solver"] = "cdiis"
    kwargs["solver_options"] = solver_options
    part = LinearISAWPart(**kwargs)
    part.do_all()
    print_results(part)
    print()

In [5]:
run_cdiis(mode="R-CDIIS", param=1e-2)

The number of electrons: 10.000003764139395
Coordinates of the atoms 
 [[ 0.     0.     0.224]
 [-0.     1.457 -0.896]
 [-0.    -1.457 -0.896]]
Atomic numbers of the atom 
 [8 1 1]

Information of integral grids.
--------------------------------------------------------------------------------
Grid size of molecular grid: 18460
************************************ Atom 0 ************************************
   |-- Radial grid size: 120
   |-- Angular grid sizes: 
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18
          18 18 18 18 18 18 26 26 26 26 38 38 38 38 38 50 50 50 50 50
          86 86 110 110 110 110 170 194 194 434 590 590 434 434 434 302 302 302 194 194
          170 110 110 110 110 110 110 110 110 110 110 110 86 50 50 18 18 18 18 18
          
************************************ Atom 1 ************************************
   |-- Radial grid size: 120
   |-

In [6]:
run_cdiis(mode="FD-CDIIS", diis_size=5)

The number of electrons: 10.000003764139395
Coordinates of the atoms 
 [[ 0.     0.     0.224]
 [-0.     1.457 -0.896]
 [-0.    -1.457 -0.896]]
Atomic numbers of the atom 
 [8 1 1]

Information of integral grids.
--------------------------------------------------------------------------------
Grid size of molecular grid: 18460
************************************ Atom 0 ************************************
   |-- Radial grid size: 120
   |-- Angular grid sizes: 
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18
          18 18 18 18 18 18 26 26 26 26 38 38 38 38 38 50 50 50 50 50
          86 86 110 110 110 110 170 194 194 434 590 590 434 434 434 302 302 302 194 194
          170 110 110 110 110 110 110 110 110 110 110 110 86 50 50 18 18 18 18 18
          
************************************ Atom 1 ************************************
   |-- Radial grid size: 120
   |-

In [7]:
run_cdiis(mode="AD-CDIIS", param=1e-4)

The number of electrons: 10.000003764139395
Coordinates of the atoms 
 [[ 0.     0.     0.224]
 [-0.     1.457 -0.896]
 [-0.    -1.457 -0.896]]
Atomic numbers of the atom 
 [8 1 1]

Information of integral grids.
--------------------------------------------------------------------------------
Grid size of molecular grid: 18460
************************************ Atom 0 ************************************
   |-- Radial grid size: 120
   |-- Angular grid sizes: 
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18
          18 18 18 18 18 18 26 26 26 26 38 38 38 38 38 50 50 50 50 50
          86 86 110 110 110 110 170 194 194 434 590 590 434 434 434 302 302 302 194 194
          170 110 110 110 110 110 110 110 110 110 110 110 86 50 50 18 18 18 18 18
          
************************************ Atom 1 ************************************
   |-- Radial grid size: 120
   |-

In [8]:
run_cdiis(mode="Roothaan")

The number of electrons: 10.000003764139395
Coordinates of the atoms 
 [[ 0.     0.     0.224]
 [-0.     1.457 -0.896]
 [-0.    -1.457 -0.896]]
Atomic numbers of the atom 
 [8 1 1]

Information of integral grids.
--------------------------------------------------------------------------------
Grid size of molecular grid: 18460
************************************ Atom 0 ************************************
   |-- Radial grid size: 120
   |-- Angular grid sizes: 
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
          6 6 6 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18
          18 18 18 18 18 18 26 26 26 26 38 38 38 38 38 50 50 50 50 50
          86 86 110 110 110 110 170 194 194 434 590 590 434 434 434 302 302 302 194 194
          170 110 110 110 110 110 110 110 110 110 110 110 86 50 50 18 18 18 18 18
          
************************************ Atom 1 ************************************
   |-- Radial grid size: 120
   |-